# (4) Model Implementation

Phase 5 of the pipeline trains two small neural networks that consume the master NPZ from Chapter 3 and predict motor performance directly from the 6-dim design vector — bypassing the per-design FEMM sweep at inference time.

**Two-network split** (rather than one big map predictor):

| Network | Input | Output | Why split |
|---|---|---|---|
| `EnvelopeMLP` (Phase 5A) | 6 design params | `(N_max, T_max)` | Predicts the absolute scale of the operating envelope |
| `MapDecoder` (Phase 5B) | 6 design params | $\eta$ on a $60\times80$ normalised grid | Predicts the *shape* of the efficiency map on a common $(N/N_\text{max}, T/T_\text{max})$ axis |

Splitting prevents the dominant peak-value variation across designs from drowning out the subtler map-shape information. The original paper does the same split.


```{figure} ../_static/figures/Fig 3_mod.png
---
name: fig-neural-network-flux
width: 85%
---
Modular architecture from the original paper: design parameters + operating point feed
a stack of fully-connected layers that produces d-q flux linkages. The open-source port
specialises this idea into two task-specific nets (`EnvelopeMLP` and `MapDecoder`).
```


## 4.1  Inputs and targets

Both networks read `outputs/dataset_train/master_dataset.npz`. The inputs are the same
6 geometry knobs; the targets differ.


In [ ]:
import numpy as np
from pathlib import Path

master = np.load(Path('data_gen/outputs/dataset_train/master_dataset.npz'),
                 allow_pickle=False)
X = master['design_params']
envelope_y = np.stack([master['N_max_rpm'], master['T_max_Nm']], axis=1)
map_y = master['eta_norm_grids']

print(f"X (design params):     {X.shape}")
print(f"envelope target:       {envelope_y.shape}  -> (N_max, T_max)")
print(f"map target:            {map_y.shape}  -> eta on (T_norm, N_norm)")
print(f"param ranges:")
for name, lo, hi in zip(master['param_names'], X.min(axis=0), X.max(axis=0)):
    print(f"  {name:<20s} {lo:8.2f} .. {hi:8.2f}")


Inputs are standardised per-fold (mean / std computed on the training rows only) before being fed to either network. Envelope targets are standardised the same way; map targets are left in $[0, 1]$ and trained with a masked MSE so cells outside the operating envelope (NaN in the source) don't contribute to the loss.


## 4.2  `EnvelopeMLP` — Phase 5A

A tiny multilayer perceptron: 6 → 32 → 32 → 2, ReLU activations, ~1.3 k parameters. Predicts the peak speed $N_\text{max}$ and peak torque $T_\text{max}$ of the operating
envelope.

Note: in the current open-source dataset every design solves for the same DC-bus voltage
and current limit, so $N_\text{max}$ is constant at 8000 RPM. The MLP collapses to a
single-target regressor for $T_\text{max}$; the $N_\text{max}$ slot is kept for
compatibility with the paper's two-output convention and for designs with varying
voltage limits.


In [ ]:
import inspect
from data_gen.ml.models import EnvelopeMLP

envelope_model = EnvelopeMLP(input_dim=6, hidden=(32, 32), output_dim=2)
print(envelope_model)
print(f"\nparameter count: {envelope_model.n_params}")


## 4.3  `MapDecoder` — Phase 5B

Predicts the normalised $\eta$ map directly from the 6-dim design vector. Architecture:

```
Linear(6 -> 64)  -> ReLU
Linear(64 -> 8*15*20=2400)  -> ReLU
reshape to (B, 8, 15, 20)
ConvTranspose2d(8 -> 4, k=4, s=2, p=1)  -> ReLU   # (B, 4, 30, 40)
ConvTranspose2d(4 -> 1, k=4, s=2, p=1)            # (B, 1, 60, 80)
sigmoid
```

About 157 k parameters. The two transposed-conv layers act as a learned upsampler;
the bilinear interpolate at the end is a no-op when the output already matches the
target $60 \times 80$ shape.


In [ ]:
from data_gen.ml.models import MapDecoder

map_model = MapDecoder(input_dim=6)
print(map_model)
print(f"\nparameter count: {map_model.n_params}")


## 4.4  Training loop and masked loss

Both networks use full-batch Adam on CPU — the dataset is small enough (2511 designs)
that mini-batching adds overhead without helping convergence.

- **EnvelopeMLP**: standard `nn.MSELoss` on standardised targets.
- **MapDecoder**: *masked* MSE — for each design's $\eta$ grid, cells outside the
  operating envelope are NaN; we replace them with zero and zero out the same cells in
  the loss with a binary mask. About 5.5% of cells are outside the envelope on average.

The masked loss implementation:


In [ ]:
from data_gen.ml.train import _masked_mse
print(inspect.getsource(_masked_mse))


Training is wrapped in `train_one_fold` (envelope) and `train_one_fold_masked` (map):


In [ ]:
from data_gen.ml import train
print(inspect.getsource(train.train_one_fold))


## 4.5  Cross-validation and standardisation

Both runners use **5-fold CV** (`kfold_indices`) — designs are shuffled once with a
fixed seed, then partitioned into 5 contiguous folds, so each design lands in exactly
one validation fold. The out-of-fold predictions (`oof_pred`) are then concatenated
to give one prediction per design, evaluated only on the fold where that design was
held out — never overlapping with the corresponding model's training set.

Input standardisation is computed from the *training* rows of each fold and reused
for the validation rows — preventing any leak of validation statistics into the
training pipeline:


In [ ]:
from data_gen.ml.data import standardise, kfold_indices
print(inspect.getsource(standardise))


## 4.6  Running the full training

The two CLI runners are self-contained drivers. They log to `data_gen/logs/`, save
fold weights, predictions, metrics, and diagnostic plots to a configurable output
directory.

```powershell
# Phase 5A — envelope net (~40 s for all 5 folds on 2511 designs)
python -m data_gen.ml.run_phase5_envelope `
    --dataset data_gen/outputs/dataset_train/master_dataset.npz `
    --out-dir data_gen/outputs/phase5_envelope_train `
    --epochs 1500 --folds 5

# Phase 5B — map decoder (~30 min for all 5 folds on 2511 designs)
python -m data_gen.ml.run_phase5_map `
    --dataset data_gen/outputs/dataset_train/master_dataset.npz `
    --out-dir data_gen/outputs/phase5_map_train `
    --epochs 1500 --folds 5 --weight-decay 1e-4
```

Each fold's weights are saved as `weights_fold{1..5}.pt`; the aggregated
out-of-fold predictions land in `predictions.npz`. Chapter 7 loads these to produce
the result figures.


## 4.7  Why these two simple networks were enough — and what we did about the η-map plateau

The original paper uses a bidirectional GRU + attention to ingest variable-length
operating-point sequences. The open-source port takes a different route:

- The dataset already produces a *fixed-shape* normalised $\eta$ grid per design.
  There is no variable-length sequence to consume.
- The mapping from 6 design params to (envelope, map) is well-behaved enough that
  a 1.3 k-param MLP and a 157 k-param ConvTranspose decoder reach OOF
  $R^2 = 0.999$ on $T_\text{max}$ and $R^2 = 0.913$ on the masked $\eta$ grid at the
  baseline architecture (5-fold CV on 2511 designs). A controlled architecture
  sweep then lifts the η-map $R^2$ to **0.973** — see §4.7.3.

The RNN/attention machinery from `03_architectures` becomes useful again when adding
the FSCW machine (the paper's second motor) or thermal-transient targets, where the
input genuinely is a sequence. For the steady-state IPM map the simpler nets win on
training time, parameter count, and final accuracy.

### 4.7.1  Where the two networks behave differently under more data

A useful diagnostic falls out when comparing the 759-design and 2511-design retrains
(same architectures, same hyperparameters, same seed). The two networks responded
very differently to the **3.3× increase in training data**:

| Network | Free parameters | $R^2$ at 759 designs | $R^2$ at 2511 designs | $\Delta$ |
|---|---|---|---|---|
| `EnvelopeMLP` ($T_\text{max}$ MAPE) | 1.3 k | 12.0% | **5.86%** | $-52\%$ |
| `MapDecoder` (masked $R^2$)         | 157 k | 0.912 | 0.913 | $+0.001$ |

The envelope predictor improved substantially with more data; the baseline map predictor did
not. This is the textbook signature of **two different regimes** on the
[bias–variance / data–capacity](https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff) plane:

- The **envelope net** is at the boundary of its data regime — adding samples reduces
  variance because the 1.3 k-param MLP has enough capacity to express the underlying
  mapping but was previously over-fitting to small-sample artifacts. More data flattens
  the validation curve.
- The **baseline map decoder** is in its **capacity regime** — the 157 k-param decoder
  has saturated what it can express about the $\eta(N, T)$ shape given the current
  architecture. Adding more FEMM samples is no longer the bottleneck. The next gain
  has to come from the *network*, not the *data*.

### 4.7.2  Architecture levers — the plan

To test the capacity hypothesis directly, we kept the 2511-design dataset fixed and
ran controlled architecture variants of `MapDecoder` with identical training
hyperparameters (1500 epochs, weight_decay=1e-4, 5-fold CV). Four levers were
identified, in roughly ascending implementation cost:

1. **Width sweep.** Scale `(8, 4)` channel widths to `(16, 8)` and `(32, 16)`. Pure
   parameter-count scale-up of the existing topology.
2. **Depth sweep.** Insert a same-resolution `Conv2d(c, c, k=3)` block between the two
   ConvTranspose stages. Adds spatial-refinement depth at no extra channel cost.
3. **Attention head** *(deferred future work)*. Replace the second `ConvTranspose`
   with a small 2-D self-attention layer over the `(15, 20)` latent grid. Designed to
   capture long-range coupling across the η map.
4. **Residual / U-Net decoder** *(deferred future work)*. Skip connections to a
   physics-derived prior (e.g. the Phase-2 envelope mask) — most data-efficient
   option but the biggest architectural change.

Levers 1–2 were executed in the architecture sweep — results in §4.7.3 below. Levers
3–4 are scoped as **future work**: now that the cheap variants hit $R^2 \approx 0.97$,
the marginal headroom for attention/U-Net is bounded above by ~0.03 in $R^2$. They
remain worth running once a follow-up iteration justifies the engineering effort.

### 4.7.3  Architecture sweep — empirical results

The width-2×, width-4×, and depth+1 variants were run on the same `dataset_train/`
splits as the baseline. All three substantially outperform baseline:

| Variant | Parameters | OOF $R^2$ | RMSE (η) | MAE (η) | Notes |
|---|---:|---:|---:|---:|---|
| baseline             | 157 k | 0.913 | 0.083 | 0.043 | Original Phase-5B topology |
| **depth+1**          | 157 k | **0.963** | **0.054** | **0.027** | Same param count + intermediate Conv2d |
| **width-2×**         | 314 k | **0.961** | **0.056** | **0.029** | bottleneck (16, 15, 20), decoder (8, 1) |
| **width-4×**         | 633 k | **0.973** | **0.046** | **0.025** | bottleneck (32, 15, 20), decoder (16, 1) |

Three observations stand out:

- **The capacity hypothesis is confirmed.** Identical training data, identical
  hyperparameters; only the architecture changed. RMSE fell by **44%** going from
  baseline to width-4×, with no extra FEMM samples.
- **depth+1 is the per-parameter winner.** A single same-resolution `Conv2d(c, c, k=3)`
  inserted between the two `ConvTranspose` stages — at **identical parameter count to
  baseline** — captures essentially the same gain as doubling all channel widths
  (width-2×). The bottleneck was spatial-refinement depth, not channel breadth.
- **Diminishing returns kick in fast.** width-2× → width-4× costs 2× the parameters
  for only +0.012 $R^2$ (0.961 → 0.973). We are approaching a new ceiling around
  $R^2 \approx 0.97$–0.98 with the current topological family (Linear-head +
  ConvTranspose decoder); attention/U-Net (§4.7.2 levers 3–4) would be needed to push
  meaningfully further.

**Headline number to carry into Chapters 5–7: $R^2 = 0.973$ from width-4×, with
depth+1 as the elegant parameter-efficient alternative.** Subsequent transfer-learning
work (Chapter 5) and uncertainty quantification (Chapter 6) will be re-evaluated on
the width-4× pretrained weights once available.


In [ ]:
# §4.7.3 — load CV metrics for the baseline + 3 architecture variants and
# render the per-variant OOF R^2 bar chart with per-fold error bars.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('data_gen/outputs')
sources = {
    'baseline (157 k)':  ROOT / 'phase5_map_train' / 'cv_metrics.csv',
    'depth+1 (157 k)':   ROOT / 'phase5_map_train_arch' / 'depth1'   / 'cv_metrics.csv',
    'width-2x (314 k)':  ROOT / 'phase5_map_train_arch' / 'width2x'  / 'cv_metrics.csv',
    'width-4x (633 k)':  ROOT / 'phase5_map_train_arch' / 'width4x'  / 'cv_metrics.csv',
}

records = []
for label, p in sources.items():
    df = pd.read_csv(p)
    folds = df[df['fold'].astype(str) != 'OVERALL'].copy()
    folds['masked_r2'] = folds['masked_r2'].astype(float)
    overall_row = df[df['fold'].astype(str) == 'OVERALL'].iloc[0]
    records.append({
        'variant': label,
        'fold_r2': folds['masked_r2'].values,
        'oof_r2': float(overall_row['masked_r2']),
        'oof_rmse': float(overall_row['masked_rmse_eta']),
    })

variants = [r['variant'] for r in records]
means = [r['fold_r2'].mean() for r in records]
stds = [r['fold_r2'].std() for r in records]
oofs = [r['oof_r2'] for r in records]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
xs = np.arange(len(variants))
colors = ['tab:gray', 'tab:green', 'tab:orange', 'tab:blue']
ax.bar(xs, means, yerr=stds, capsize=6, color=colors, alpha=0.85, edgecolor='black')
for i, v in enumerate(oofs):
    ax.text(xs[i], means[i] + max(stds[i], 0.005) + 0.005,
            f'OOF\n{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(xs)
ax.set_xticklabels(variants, rotation=12)
ax.set_ylabel(r'masked $R^2$ on $\eta$ map')
ax.set_title('MapDecoder architecture sweep  (5-fold CV, 2511 designs)\n'
             'bars = per-fold mean ± 1 std;  numbers = pooled OOF $R^2$')
ax.set_ylim(0.88, 1.00)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nVariant summary:')
print(pd.DataFrame([{
    'variant': r['variant'],
    'fold_mean_R2': round(r['fold_r2'].mean(), 4),
    'fold_std_R2':  round(r['fold_r2'].std(), 4),
    'OOF_R2':       round(r['oof_r2'], 4),
    'OOF_RMSE':     round(r['oof_rmse'], 4),
} for r in records]).to_string(index=False))

:::{admonition} Future work — §4.7.2 lever 3: 2-D self-attention head
:class: tip
**Expected wall:** ~2–4 hours per 5-fold CV on CPU (~30–50 min per fold for a small (15 × 20) self-attention head replacing decoder[-1]).
**Expected advantage:** pushes the current $R^2 \approx 0.97$ ceiling. Marginal headroom is bounded above by ~0.03 in R² given the diminishing-returns curve observed across baseline → width-2× → width-4× (+0.06, +0.05, +0.01 incrementally). Attention's structural advantage is *long-range spatial coupling* across the η map — useful if low-speed/high-torque corner cells share information with high-speed/low-torque cells through the rotor-flux dependence. Sweep checks that hypothesis.
**Implementation hook:** add a new entry to `MAPDECODER_PRESETS` in `data_gen/ml/models.py`, e.g. `'attention'`, replacing the second `ConvTranspose2d` with a small `nn.MultiheadAttention(embed_dim=c_mid, num_heads=4)` over the flattened latent grid.
:::

:::{admonition} Future work — §4.7.2 lever 4: U-Net residual decoder with envelope-mask prior
:class: tip
**Expected wall:** ~2–4 hours per 5-fold CV on CPU (similar order to width-4× since the parameter count is in the same ballpark).
**Expected advantage:** the most data-efficient option in the lever list. A U-Net-style skip-connect from the Phase-2 envelope mask (which the dataset already produces per design) injects a physics-derived prior at the decoder bottleneck, potentially closing the boundary-cell error pattern visible in Chapter 6's per-cell residual map. Could push past $R^2 = 0.98$ at the same parameter count as width-4×; even if it merely matches width-4× the residual structure should change — boundary cells lose their bias.
**Implementation hook:** modify `MapDecoder.forward` to take a second input (the envelope mask), concatenate it as an extra channel after the first `ConvTranspose`, and add a residual addition before the final sigmoid.
:::